# `klygo.config` (`io.py`): Configuration file I/O, conversion, and export

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IchigoMazone/klygo/blob/v2-release/tutorials/config/Document_Config_Io.ipynb)


Read, write, convert, and export structured configuration files (YAML, JSON, TOML, INI, ENV) with dot-accessible `Box` objects, `./` root path expansion, and lossless TOML `None` preservation.

Run this notebook from top to bottom. It creates only temporary example files.

In [ ]:
!pip install --upgrade --no-cache-dir git+https://github.com/IchigoMazone/klygo.git@v2-release

## Prepare sample files

Run this setup cell again at any time to reset the temporary workspace.

In [1]:
from tempfile import TemporaryDirectory
from klygo import config, files

# Run this cell again to replace only this notebook's temporary workspace.
if "tutorial_workspace" in globals():
    tutorial_workspace.cleanup()
tutorial_workspace = TemporaryDirectory(prefix="klygo-config-")
root = files.path(tutorial_workspace.name)
print("Example directory:", root)

Example directory: C:\Users\Mine\AppData\Local\Temp\klygo-config-s4ldjct7


## Jump to an API

[`load`](#load) · [`save`](#save) · [`convert`](#convert) · [`export`](#export)

<a id="load"></a>

### `config.load`

Load a configuration file as a dot-accessible `Box`.

**Signature:** `config.load(path: str | pathlib.Path, verbose: bool = True) -> box.Box`

In [2]:
settings_file = root / "train.yaml"
config.save(
    settings_file,
    {"default": {"root": "/workspace/data"}, "dataset": {"images": "./images"}, "model": {"batch": 16}},
    overwrite=True,
    verbose=False,
)
loaded = config.load(settings_file, verbose=False)
print("Batch:", loaded.model.batch)
print("Expanded images path:", loaded.dataset.images)

Batch: 16
Expanded images path: \workspace\data\images


In [3]:
# Error example: loading a missing file or a non-mapping JSON list
bad_json = root / "list.json"
files.save(bad_json, [1, 2, 3], overwrite=True, verbose=False)
for target in (root / 'missing.yaml', bad_json):
    try:
        config.load(target, verbose=False)
    except (FileNotFoundError, TypeError) as exc:
        print(type(exc).__name__, "->", exc)

FileNotFoundError -> Path does not exist: C:\Users\Mine\AppData\Local\Temp\klygo-config-s4ldjct7\missing.yaml
TypeError -> configuration root must be a mapping


<a id="save"></a>

### `config.save`

Save configuration data using the destination filename extension.

**Signature:** `config.save(path: str | pathlib.Path, data: Mapping[str, Any] | box.Box, overwrite: bool = False, verbose: bool = True) -> pathlib.Path`

In [4]:
from klygo import models

flags = models.flags(model=False, processor=True, post=True)
schema = models.metadata(flags)
toml_path = config.save(root / "model_schema.toml", schema, overwrite=True, verbose=False)
round_trip = config.load(toml_path, verbose=False)
print("Saved to:", files.name(toml_path))
print("Preserved None in TOML:", round_trip["class"] is None, "| flags.model:", round_trip.flags.model)

Saved to: model_schema.toml
Preserved None in TOML: True | flags.model: False


In [5]:
# Error example: overwrite protection and non-mapping input
try:
    config.save(toml_path, {"debug": True}, overwrite=False, verbose=False)
except FileExistsError as exc:
    print("FileExistsError ->", exc)

try:
    config.save(root / "invalid.json", ["not", "a", "mapping"], verbose=False)
except TypeError as exc:
    print("TypeError ->", exc)

FileExistsError -> File already exists: C:\Users\Mine\AppData\Local\Temp\klygo-config-s4ldjct7\model_schema.toml. Use overwrite=True to replace it.
TypeError -> data must be Mapping, Box, got list


<a id="convert"></a>

### `config.convert`

Convert a configuration file to another supported format.

**Signature:** `config.convert(source: str | pathlib.Path, target: str | pathlib.Path, overwrite: bool = False, verbose: bool = True) -> pathlib.Path`

In [6]:
converted_json = config.convert(settings_file, root / "train.json", overwrite=True, verbose=False)
print("Converted file:", files.name(converted_json))
print("Reloaded batch:", config.load(converted_json, verbose=False).model.batch)

Converted file: train.json
Reloaded batch: 16


In [7]:
# Error example: converting a missing source file
try:
    config.convert(root / "no_such.yaml", root / "out.json", verbose=False)
except FileNotFoundError as exc:
    print("FileNotFoundError ->", exc)

FileNotFoundError -> Path does not exist: C:\Users\Mine\AppData\Local\Temp\klygo-config-s4ldjct7\no_such.yaml


<a id="export"></a>

### `config.export`

Export a configuration object or existing file to a destination file.

**Signature:** `config.export(source: str | pathlib.Path | Mapping[str, Any] | box.Box, target: str | pathlib.Path, overwrite: bool = False, verbose: bool = True) -> pathlib.Path`

In [8]:
from_mem = config.export({"experiment": "baseline", "seed": 42}, root / "exp.toml", overwrite=True, verbose=False)
from_file = config.export(from_mem, root / "exp.yaml", overwrite=True, verbose=False)
print("Exported from dict:", files.name(from_mem))
print("Exported from file:", files.name(from_file), "->", config.load(from_file, verbose=False).to_dict())

Exported from dict: exp.toml
Exported from file: exp.yaml -> {'experiment': 'baseline', 'seed': 42}


In [9]:
# Error example: unsupported source type
try:
    config.export(12345, root / "bad.json", verbose=False)
except TypeError as exc:
    print("TypeError ->", exc)

TypeError -> source must be a path, mapping, or Box


## Put it together

In [10]:
pipeline_yaml = config.save(
    root / "pipeline.yaml",
    {"default": {"root": "/data/project"}, "paths": {"train": "./train"}, "seed": None},
    overwrite=True,
    verbose=False,
)
pipeline_toml = config.convert(pipeline_yaml, root / "pipeline.toml", overwrite=True, verbose=False)
reloaded = config.load(pipeline_toml, verbose=False)
print("Resolved train path:", reloaded.paths.train, "| seed is None:", reloaded.seed is None)

Resolved train path: \data\project\train | seed is None: True


## Remember

- `load()` expands `./` strings when `default.root` is present and restores TOML `None` sentinel values saved by `klygo.config`.
- `save()`, `convert()`, and `export()` infer the output format from the target file extension and require `overwrite=True` to replace an existing file.
- Next in `klygo.config`: continue with [Document_Config_Creation.ipynb](Document_Config_Creation.ipynb) or return to [Document_Config_Overview.ipynb](Document_Config_Overview.ipynb).
- Related Klygo modules: [`klygo.files`](../files/Document_Files_Overview.ipynb) · [`klygo.archive`](../archive/Document_Archive_Overview.ipynb).